# Fazer upload das tabelas de indicadores no Neon database

In [1]:
# imports

import os
from pathlib import Path

import pandas as pd
import psycopg
from dotenv import load_dotenv

In [2]:
# instâncias

load_dotenv()

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRUSTED_DIR = PROJECT_ROOT / "data" / "trusted"

DATABASE_URL = os.getenv("DATABASE_URL")


TABELAS = {
    "emp_var_rate_mensal": TRUSTED_DIR / "emp_var_rate_mensal.parquet",
    "cons_price_idx_mensal": TRUSTED_DIR / "cons_price_idx_mensal.parquet",
    "cons_conf_idx_mensal": TRUSTED_DIR / "cons_conf_idx_mensal.parquet",
    "euribor3m_mensal": TRUSTED_DIR / "euribor3m_mensal.parquet",
    "nr_employed_mensal": TRUSTED_DIR / "nr_employed_mensal.parquet"
}


CREATE_SQL = {
    "emp_var_rate_mensal": """
        CREATE TABLE IF NOT EXISTS emp_var_rate_mensal (
            month TEXT NOT NULL,
            year INTEGER NOT NULL,
            emp_var_rate DOUBLE PRECISION NOT NULL,
            PRIMARY KEY (month, year)
        )
    """,

    "cons_price_idx_mensal": """
        CREATE TABLE IF NOT EXISTS cons_price_idx_mensal (
            month TEXT NOT NULL,
            year INTEGER NOT NULL,
            cons_price_idx DOUBLE PRECISION NOT NULL,
            PRIMARY KEY (month, year)
        )
    """,

    "cons_conf_idx_mensal": """
        CREATE TABLE IF NOT EXISTS cons_conf_idx_mensal (
            month TEXT NOT NULL,
            year INTEGER NOT NULL,
            cons_conf_idx DOUBLE PRECISION NOT NULL,
            PRIMARY KEY (month, year)
        )
    """,

    "euribor3m_mensal": """
        CREATE TABLE IF NOT EXISTS euribor3m_mensal (
            month TEXT NOT NULL,
            year INTEGER NOT NULL,
            month_position TEXT NOT NULL,
            euribor3m DOUBLE PRECISION NOT NULL,
            PRIMARY KEY (month, year, month_position)
        )
    """,

    "nr_employed_mensal": """
        CREATE TABLE IF NOT EXISTS nr_employed_mensal (
            month TEXT NOT NULL,
            year INTEGER NOT NULL,
            nr_employed DOUBLE PRECISION NOT NULL,
            PRIMARY KEY (month, year)
        )
    """
}


In [4]:
# fazer upload

with psycopg.connect(DATABASE_URL) as conn:

    with conn.cursor() as cur:

        for table_name, file_path in TABELAS.items():

            df = pd.read_parquet(
                file_path
            )

            # Ajuste caso o parquet antigo ainda use outro nome.
            if (
                table_name == "euribor3m_mensal"
                and "periodo_mes" in df.columns
                and "month_position" not in df.columns
            ):
                df = df.rename(
                    columns={
                        "periodo_mes":
                        "month_position"
                    }
                )

            cur.execute(
                CREATE_SQL[table_name]
            )

            cur.execute(
                f"TRUNCATE TABLE {table_name}"
            )

            columns = list(df.columns)

            placeholders = ", ".join(
                ["%s"] * len(columns)
            )

            column_names = ", ".join(
                columns
            )

            insert_sql = (
                f"INSERT INTO {table_name} "
                f"({column_names}) "
                f"VALUES ({placeholders})"
            )

            rows = [
                tuple(row)
                for row in df.itertuples(
                    index=False,
                    name=None
                )
            ]

            cur.executemany(
                insert_sql,
                rows
            )

            print(
                f"{table_name}: {len(rows)} registros enviados."
            )

    conn.commit()

print(
    "Indicadores persistidos no Neon com sucesso."
)

emp_var_rate_mensal: 26 registros enviados.
cons_price_idx_mensal: 26 registros enviados.
cons_conf_idx_mensal: 26 registros enviados.
euribor3m_mensal: 78 registros enviados.
nr_employed_mensal: 26 registros enviados.
Indicadores persistidos no Neon com sucesso.
